In [1]:
import pandas as pd

DATA_PATH = "/lakehouse/default/Files/raw_comexstat/"

ARQUIVO_PARQUET = "comexstat_municipios_1997_2026.parquet"

def load_comexstat_data():
    """Load and prepare Comexstat data for Osasco municipality."""
    file_path = DATA_PATH + ARQUIVO_PARQUET
    df = pd.read_parquet(file_path)
    
    # Data type conversions
    df = df.astype({
        "VL_FOB": "int64",
        "SH4": "str",
        "CO_ANO": "int32",
    })
    
    # Filter for Osasco municipality (code 3434401)
    df_oz = df.query("CO_MUN == 3434401").copy()
    
    # Load SH classification data
    aux_sh = pd.read_csv(DATA_PATH + "NCM_SH.csv", sep=";", encoding="latin1")
    aux_sh = aux_sh.rename(columns={"CO_SH4": "SH4"}).astype({"SH4": "str"})
    aux_sh = (
        aux_sh[["SH4", "NO_SH4_POR", "NO_SH2_POR", "NO_SEC_POR"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    
    # Merge with descriptions
    df_oz_desc = df_oz.merge(aux_sh, on="SH4", how="left")
    df_oz_desc.columns = df_oz_desc.columns.str.lower()
    
    # Select and organize columns
    df_oz_desc = df_oz_desc[
        [
            "co_ano",
            "co_mes",
            "sh4",
            "co_mun",
            "vl_fob",
            "tipo",
            "no_sh4_por",
            "no_sh2_por",
            "no_sec_por",
        ]
    ].copy()
    
    return df_oz_desc

df = load_comexstat_data()

StatementMeta(, b8cd3b16-c5e8-4c7a-9f48-203da356ed1b, 3, Finished, Available, Finished, False)

In [4]:
sdf = spark.createDataFrame(df)
(
    sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_osasco_comexstat")
)

StatementMeta(, b8cd3b16-c5e8-4c7a-9f48-203da356ed1b, 6, Finished, Available, Finished, False)